In [1]:
!pip install -q scikit-learn tensorflow joblib nltk Sastrawi tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.2 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import pickle
import string
import warnings
import numpy as np
import pandas as pd
import joblib

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("Import selesai.")

Import selesai.


In [3]:
with open('models/model_metadata.json', 'r') as f:
    metadata = json.load(f)

print("=== Model Metadata ===")
print(json.dumps(metadata, indent=2))

BEST_MODEL_TYPE = metadata['best_model_type']   # 'ml' atau 'dl'
BEST_MODEL_NAME = metadata['best_model_name']
LABELS          = metadata['labels']
print(f"\nModel terbaik : {BEST_MODEL_NAME}")
print(f"Tipe          : {BEST_MODEL_TYPE}")
print(f"Label kelas   : {LABELS}")

=== Model Metadata ===
{
  "best_model_name": "BiGRU",
  "best_model_type": "dl",
  "test_accuracy": 0.9054,
  "f1_macro": 0.8292,
  "labels": [
    "negative",
    "neutral",
    "positive"
  ],
  "num_classes": 3,
  "preprocessing_notes": "clean_text dari 02_eda_cleaning_labeling.ipynb",
  "required_artifacts": [
    "/content/models/best_model.keras",
    "/content/models/tokenizer_best.pkl",
    "/content/models/label_encoder.joblib"
  ],
  "dl_config": {
    "max_features": 10000,
    "max_len": 50,
    "embed_dim": 128
  }
}

Model terbaik : BiGRU
Tipe          : dl
Label kelas   : ['negative', 'neutral', 'positive']


In [4]:
# Load label encoder (digunakan oleh semua skema)
le = joblib.load('models/label_encoder.joblib')
print(f"Label encoder dimuat. Kelas: {le.classes_.tolist()}")

if BEST_MODEL_TYPE == 'ml':
    # Logistic Regression
    model    = joblib.load('models/best_model.joblib')
    vectorizer = joblib.load('models/vectorizer_tfidf.joblib')
    tokenizer_dl = None
    MAX_LEN  = None
    print("Model ML (Logistic Regression) dimuat.")
else:
    # Deep Learning (BiLSTM atau BiGRU)
    model    = load_model('models/best_model.keras')
    with open('models/tokenizer_best.pkl', 'rb') as f:
        tokenizer_dl = pickle.load(f)
    vectorizer = None
    with open('models/dl_config.json', 'r') as f:
        dl_config = json.load(f)
    MAX_LEN = dl_config['max_len']
    print(f"Model DL ({BEST_MODEL_NAME}) dimuat. MAX_LEN={MAX_LEN}")

Label encoder dimuat. Kelas: ['negative', 'neutral', 'positive']
Model DL (BiGRU) dimuat. MAX_LEN=50


In [5]:
# --- Inisialisasi ---
factory = StemmerFactory()
stemmer = factory.create_stemmer()

id_stopwords = set(stopwords.words('indonesian'))
en_stopwords = set(stopwords.words('english'))
all_stopwords = id_stopwords.union(en_stopwords)

# Load kamus slang
SLANG_URL = "https://raw.githubusercontent.com/adeariniputri/text-preprocesing/master/slang.csv"
slang_dict = {}
try:
    slang_df = pd.read_csv(SLANG_URL, header=None, names=['slang', 'formal'])
    slang_dict = dict(zip(slang_df['slang'].str.lower(), slang_df['formal'].str.lower()))
    print(f"Kamus slang dimuat: {len(slang_dict)} entri")
except Exception as e:
    print(f"Kamus slang tidak tersedia ({e}), normalisasi slang dilewati.")

# --- Fungsi per tahap ---
def remove_mentions(text):    return re.sub(r'@\w+', ' ', text)
def remove_hashtags(text):    return re.sub(r'#\w+', ' ', text)
def remove_rt(text):          return re.sub(r'\bRT\b', ' ', text)
def remove_urls(text):        return re.sub(r'http\S+|www\.\S+', ' ', text)
def remove_numbers(text):     return re.sub(r'\d+', ' ', text)
def remove_punctuation(text): return text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
def remove_newlines(text):    return text.replace('\n', ' ').replace('\r', ' ')
def normalize_whitespace(text): return re.sub(r'\s+', ' ', text).strip()
def case_folding(text):       return text.lower()

def normalize_slang(text):
    if not slang_dict:
        return text
    return ' '.join([slang_dict.get(w, w) for w in text.split()])

def tokenize(text):           return word_tokenize(text)
def remove_stopwords(tokens): return [t for t in tokens if t not in all_stopwords and len(t) > 1]
def stem_tokens(tokens):      return [stemmer.stem(t) for t in tokens]

def preprocess_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text)
    text = remove_mentions(text)
    text = remove_hashtags(text)
    text = remove_rt(text)
    text = remove_urls(text)
    text = remove_numbers(text)
    text = remove_punctuation(text)
    text = remove_newlines(text)
    text = normalize_whitespace(text)
    text = case_folding(text)
    text = normalize_slang(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = stem_tokens(tokens)
    return ' '.join(tokens)

print("Fungsi preprocessing ready.")

Kamus slang dimuat: 1481 entri
Fungsi preprocessing ready.


In [6]:
def predict_sentiment(raw_text):
    """
    Menerima teks mentah, mengembalikan (label, confidence).
    """
    clean = preprocess_text(raw_text)

    if BEST_MODEL_TYPE == 'ml':
        vec   = vectorizer.transform([clean])
        proba = model.predict_proba(vec)[0]
        idx   = np.argmax(proba)
    else:
        seq   = pad_sequences(
                    tokenizer_dl.texts_to_sequences([clean]),
                    maxlen=MAX_LEN, padding='post', truncating='post')
        proba = model.predict(seq, verbose=0)[0]
        idx   = np.argmax(proba)

    label      = le.inverse_transform([idx])[0]
    confidence = float(proba[idx])
    return label, confidence, clean

# Uji cepat satu teks
label, conf, clean = predict_sentiment("Aplikasi ini sangat membantu saya membayar pajak")
print(f"Label      : {label}")
print(f"Confidence : {conf:.4f}")
print(f"Clean text : {clean}")

Label      : negative
Confidence : 0.9960
Clean text : aplikasi bantu bayar pajak


In [7]:
sample_texts = [
    "Aplikasi ini sangat membantu dan mudah digunakan untuk bayar pajak",
    "Error terus tidak bisa login, sangat mengecewakan",
    "Sudah coba download aplikasinya",
    "Bagus sekali, pelayanan cepat dan responsif",
    "Aplikasi sering crash dan tidak bisa dibuka sama sekali",
    "Biasa saja, tidak ada yang istimewa dari aplikasi ini",
    "Terima kasih, pembayaran pajak jadi lebih mudah",
    "Tolong diperbaiki, banyak bug dan error",
    "Fitur lengkap tapi agak lambat",
    "Tidak bisa masuk ke akun saya sudah berhari-hari",
]

results = []
for text in sample_texts:
    label, conf, clean = predict_sentiment(text)
    results.append({
        'raw_text': text,
        'clean_text': clean,
        'predicted_sentiment': label,
        'confidence': round(conf, 4)
    })

result_df = pd.DataFrame(results)
print(result_df.to_string(index=False))

                                                          raw_text                       clean_text predicted_sentiment  confidence
Aplikasi ini sangat membantu dan mudah digunakan untuk bayar pajak aplikasi bantu mudah bayar pajak            positive      0.8023
                 Error terus tidak bisa login, sangat mengecewakan              ganggu masuk kecewa            negative      0.9999
                                   Sudah coba download aplikasinya              coba unduh aplikasi            negative      0.9709
                       Bagus sekali, pelayanan cepat dan responsif      bagus layan cepat responsif            positive      0.9999
           Aplikasi sering crash dan tidak bisa dibuka sama sekali             aplikasi tabrak buka            negative      0.9678
             Biasa saja, tidak ada yang istimewa dari aplikasi ini                istimewa aplikasi            negative      0.9951
                   Terima kasih, pembayaran pajak jadi lebih mudah   terima 

In [8]:
# Tampilkan dengan format yang lebih rapi
pd.set_option('display.max_colwidth', 60)
result_df[['raw_text', 'predicted_sentiment', 'confidence']]

,raw_text,predicted_sentiment,confidence
0,Aplikasi ini sangat membantu dan mudah digunakan untuk b...,positive,0.8023
1,"Error terus tidak bisa login, sangat mengecewakan",negative,0.9999
2,Sudah coba download aplikasinya,negative,0.9709
3,"Bagus sekali, pelayanan cepat dan responsif",positive,0.9999
4,Aplikasi sering crash dan tidak bisa dibuka sama sekali,negative,0.9678
5,"Biasa saja, tidak ada yang istimewa dari aplikasi ini",negative,0.9951
6,"Terima kasih, pembayaran pajak jadi lebih mudah",positive,0.9996
7,"Tolong diperbaiki, banyak bug dan error",positive,0.8103
8,Fitur lengkap tapi agak lambat,positive,0.9639
9,Tidak bisa masuk ke akun saya sudah berhari-hari,negative,0.9991


In [9]:
# Distribusi prediksi pada sampel
print("Distribusi prediksi pada sampel teks:")
print(result_df['predicted_sentiment'].value_counts())

Distribusi prediksi pada sampel teks:
predicted_sentiment
positive    5
negative    5
Name: count, dtype: int64


In [16]:
custom_text = "gua terbantu banget sama app ini"

label, conf, clean = predict_sentiment(custom_text)
print(f"Input      : {custom_text}")
print(f"Clean      : {clean}")
print(f"Sentimen   : {label.upper()}")
print(f"Confidence : {conf:.2%}")

Input      : gua terbantu banget sama app ini
Clean      : bantu aplikasi
Sentimen   : NEGATIVE
Confidence : 80.89%


In [17]:
sample_texts = [
    "gua terbantu banget sama app ini",
    "aplikasinya bagus dan mudah dipakai",
    "fiturnya lengkap tapi kadang agak lemot",
    "jelek banget aplikasinya sering error",
    "saya suka tampilannya simpel dan nyaman",
    "aplikasi ini tidak membantu sama sekali",
    "lumayan sih tapi masih banyak bug",
    "mantap banget, sangat berguna untuk kebutuhan saya",
    "kecewa karena sering crash saat dibuka",
    "biasa aja, tidak terlalu bagus tapi juga tidak buruk"
]

for text in sample_texts:
    label, conf, clean = predict_sentiment(text)

    print("=" * 60)
    print(f"Input      : {text}")
    print(f"Clean      : {clean}")
    print(f"Sentimen   : {label.upper()}")
    print(f"Confidence : {conf:.2%}")

Input      : gua terbantu banget sama app ini
Clean      : bantu aplikasi
Sentimen   : NEGATIVE
Confidence : 80.89%
Input      : aplikasinya bagus dan mudah dipakai
Clean      : aplikasi bagus mudah pakai
Sentimen   : POSITIVE
Confidence : 98.60%
Input      : fiturnya lengkap tapi kadang agak lemot
Clean      : fiturnya lengkap kadang lambat
Sentimen   : POSITIVE
Confidence : 99.95%
Input      : jelek banget aplikasinya sering error
Clean      : jelek aplikasi ganggu
Sentimen   : NEGATIVE
Confidence : 100.00%
Input      : saya suka tampilannya simpel dan nyaman
Clean      : suka tampil simpel nyaman
Sentimen   : POSITIVE
Confidence : 99.99%
Input      : aplikasi ini tidak membantu sama sekali
Clean      : aplikasi bantu
Sentimen   : NEGATIVE
Confidence : 64.40%
Input      : lumayan sih tapi masih banyak bug
Clean      : lumayan sih virus
Sentimen   : POSITIVE
Confidence : 62.76%
Input      : mantap banget, sangat berguna untuk kebutuhan saya
Clean      : mantap guna butuh
Sentimen   : 

In [19]:
!zip -r /content/04_inference.zip /content/

  adding: content/ (stored 0%)
  adding: content/.config/ (stored 0%)
  adding: content/.config/gce (stored 0%)
  adding: content/.config/.last_survey_prompt.yaml (stored 0%)
  adding: content/.config/.last_update_check.json (deflated 22%)
  adding: content/.config/config_sentinel (stored 0%)
  adding: content/.config/active_config (stored 0%)
  adding: content/.config/configurations/ (stored 0%)
  adding: content/.config/configurations/config_default (deflated 15%)
  adding: content/.config/default_configs.db (deflated 98%)
  adding: content/.config/.last_opt_in_prompt.yaml (stored 0%)
  adding: content/.config/logs/ (stored 0%)
  adding: content/.config/logs/2026.05.21/ (stored 0%)
  adding: content/.config/logs/2026.05.21/13.26.27.601334.log (deflated 58%)
  adding: content/.config/logs/2026.05.21/13.26.44.509737.log (deflated 58%)
  adding: content/.config/logs/2026.05.21/13.26.07.261908.log (deflated 92%)
  adding: content/.config/logs/2026.05.21/13.26.59.793897.log (deflated 56%)